# ForensiX CLIP-style image encoder training (Kaggle T4 x2)

This notebook trains a CLIP-style dual encoder **from random initialization**. It expects an image/caption manifest with at least 100,000 rows; the images are not embedded in this repository. The manifest must contain `image` and `caption` columns, where `image` is an absolute path or a path relative to `DATA_ROOT`.

The output is a normalized image encoder exported as `forensix_clip_image_encoder.onnx`, suitable for the ForensiX `FORENSIX_IMAGE_EMBEDDING_ONNX_MODEL` adapter. Training from scratch is substantially slower and less accurate than starting from a pretrained CLIP checkpoint.

In [ ]:
!pip -q install open_clip_torch accelerate pandas pillow scikit-learn onnx onnxruntime-gpu datasets requests tqdm matplotlib


In [ ]:
from pathlib import Path
import os, time, math, json, random, hashlib, shutil
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from accelerate import Accelerator
from datasets import load_dataset
from tqdm.auto import tqdm
import requests
import open_clip

# Change these paths for the Kaggle dataset attached to the notebook.
DATA_ROOT = Path(os.environ.get('FORENSIX_DATA_ROOT', '/kaggle/input/forensix-100k'))
MANIFEST = Path(os.environ.get('FORENSIX_MANIFEST', str(DATA_ROOT / 'pairs.csv')))
OUTPUT_DIR = Path('/kaggle/working/forensix_clip_output')
AUTO_MANIFEST_IF_MISSING = True
AUTO_CAPTION_TEMPLATE = 'forensic gallery image from dataset {dataset}'
USE_HUGGINGFACE_IF_LOCAL_EMPTY = True
HF_DATASET_CANDIDATES = [
    {'path': 'phiyodr/coco2017', 'name': 'coco-karpathy-long', 'split': 'train'},
    {'path': 'phiyodr/coco2017', 'name': 'coco-karpathy', 'split': 'train'},
    {'path': 'MagiBoss/COCO-Image-Captioning', 'name': None, 'split': 'train'},
    {'path': 'lmms-lab/COCO-Caption', 'name': None, 'split': 'train'},
]
HF_IMAGE_DIR = OUTPUT_DIR / 'hf_images'
HF_MAX_IMAGES = 100_000
HF_STREAMING = True
HTTP_TIMEOUT_SECONDS = 20
MIN_IMAGES = 100_000
MODEL_NAME = 'ViT-B-32'
IMAGE_SIZE = 224
EPOCHS = 10
BATCH_PER_GPU = 128
GRAD_ACCUM_STEPS = 1
LR = 5e-4
WEIGHT_DECAY = 0.1
TEMPERATURE = 0.07
NUM_WORKERS = min(8, os.cpu_count() or 2)
SEED = 1337
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT:', DATA_ROOT)
print('MANIFEST:', MANIFEST)
print('CUDA probe skipped here so Accelerate can launch GPU workers cleanly.')
print('Kaggle should show the selected GPU in the notebook sidebar.')


## Manifest contract

Best path: create `/kaggle/input/forensix-100k/pairs.csv` with rows such as:

```csv
image,caption
images/000001.jpg,person standing beside a vehicle
images/000002.jpg,indoor room with a table
```

Captions should describe the image. Repeating generic captions will train a weak embedding space.

If `pairs.csv` is missing, the next cell can auto-discover images under `/kaggle/input` and create a bootstrap manifest at `/kaggle/working/forensix_clip_output/auto_pairs.csv`. That is useful for smoke tests and building the ForensiX pipeline, but it is not a substitute for real captions.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
CAPTION_FIELDS = ['caption', 'text', 'sentence', 'prompt', 'description']
CAPTION_LIST_FIELDS = ['captions', 'sentences', 'answers']
IMAGE_FIELDS = ['image', 'jpg', 'png']
URL_FIELDS = ['url', 'image_url', 'coco_url', 'flickr_url']

def discover_images(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(path for path in root.rglob('*') if path.suffix.lower() in IMAGE_EXTENSIONS)

def first_text(value):
    if value is None:
        return None
    if isinstance(value, str):
        return value.strip() or None
    if isinstance(value, (list, tuple)):
        for item in value:
            text = first_text(item)
            if text:
                return text
    if isinstance(value, dict):
        for key in CAPTION_FIELDS:
            text = first_text(value.get(key))
            if text:
                return text
    return None

def caption_from_row(row):
    for key in CAPTION_FIELDS:
        text = first_text(row.get(key))
        if text:
            return text
    for key in CAPTION_LIST_FIELDS:
        text = first_text(row.get(key))
        if text:
            return text
    return None

def image_file_name(prefix, index, extension='.jpg'):
    extension = extension.lower()
    if extension not in IMAGE_EXTENSIONS:
        extension = '.jpg'
    digest = hashlib.sha1(f'{prefix}-{index}'.encode()).hexdigest()[:16]
    return HF_IMAGE_DIR / f'{prefix}_{index:06d}_{digest}{extension}'

def save_pil_image(image, prefix, index):
    path = image_file_name(prefix, index, '.jpg')
    if not path.exists():
        image.convert('RGB').save(path, format='JPEG', quality=92)
    return path

def download_image(url, prefix, index):
    extension = Path(str(url).split('?')[0]).suffix.lower() or '.jpg'
    path = image_file_name(prefix, index, extension)
    if path.exists() and path.stat().st_size > 0:
        return path
    response = requests.get(url, timeout=HTTP_TIMEOUT_SECONDS)
    response.raise_for_status()
    path.write_bytes(response.content)
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception:
        path.unlink(missing_ok=True)
        return None
    return path

def image_from_row(row, prefix, index):
    for key in IMAGE_FIELDS:
        value = row.get(key)
        if isinstance(value, Image.Image):
            return save_pil_image(value, prefix, index)
        if isinstance(value, dict):
            if isinstance(value.get('path'), str) and Path(value['path']).exists():
                return Path(value['path'])
            if value.get('bytes'):
                path = image_file_name(prefix, index, '.jpg')
                path.write_bytes(value['bytes'])
                return path
        if isinstance(value, str):
            if value.startswith(('http://', 'https://')):
                found = download_image(value, prefix, index)
                if found:
                    return found
            path = Path(value)
            if path.exists():
                return path
    for key in URL_FIELDS:
        value = row.get(key)
        if isinstance(value, str) and value.startswith(('http://', 'https://')):
            found = download_image(value, prefix, index)
            if found:
                return found
    return None

def load_hf_candidate(candidate):
    kwargs = {'split': candidate.get('split', 'train'), 'streaming': HF_STREAMING}
    if candidate.get('name'):
        return load_dataset(candidate['path'], candidate['name'], **kwargs)
    return load_dataset(candidate['path'], **kwargs)

def build_huggingface_manifest():
    HF_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    errors = []
    for candidate in HF_DATASET_CANDIDATES:
        label = candidate['path'] + (f"/{candidate['name']}" if candidate.get('name') else '')
        rows, seen = [], set()
        print(f'Trying Hugging Face dataset: {label}')
        try:
            dataset = load_hf_candidate(candidate)
            for index, row in enumerate(tqdm(dataset, total=HF_MAX_IMAGES, desc=label)):
                caption = caption_from_row(row)
                if not caption:
                    continue
                path = image_from_row(row, candidate['path'].split('/')[-1].lower(), index)
                if not path or str(path) in seen:
                    continue
                seen.add(str(path))
                rows.append({'image': str(path), 'caption': caption})
                if len(rows) >= HF_MAX_IMAGES:
                    break
        except Exception as exc:
            errors.append(f'{label}: {type(exc).__name__}: {exc}')
            print(errors[-1])
            continue
        if rows:
            frame = pd.DataFrame(rows).drop_duplicates('image').reset_index(drop=True)
            hf_manifest = OUTPUT_DIR / 'hf_pairs.csv'
            frame.to_csv(hf_manifest, index=False)
            print(f'Created {hf_manifest} from {label} with {len(frame):,} image-caption pairs.')
            return frame
    raise FileNotFoundError('Could not build a Hugging Face manifest. Last errors: ' + ' | '.join(errors[-3:]))

def build_auto_manifest():
    kaggle_input = Path('/kaggle/input')
    search_roots = [DATA_ROOT] if DATA_ROOT.exists() else []
    if kaggle_input.exists():
        search_roots.extend(path for path in kaggle_input.iterdir() if path.is_dir() and path != DATA_ROOT)
    rows = []
    for root in search_roots:
        dataset_name = root.name.replace('-', ' ').replace('_', ' ')
        for image_path in discover_images(root):
            rows.append({'image': str(image_path), 'caption': AUTO_CAPTION_TEMPLATE.format(dataset=dataset_name)})
    if rows:
        frame = pd.DataFrame(rows).drop_duplicates('image').reset_index(drop=True)
        auto_manifest = OUTPUT_DIR / 'auto_pairs.csv'
        frame.to_csv(auto_manifest, index=False)
        print(f'MANIFEST was missing, so I created {auto_manifest} with {len(frame):,} images.')
        print('For strong CLIP quality, replace generic captions with real image captions later.')
        return frame
    if USE_HUGGINGFACE_IF_LOCAL_EMPTY:
        print('No local Kaggle images found. Downloading an image-caption dataset from Hugging Face instead.')
        return build_huggingface_manifest()
    raise FileNotFoundError('No images found under DATA_ROOT or /kaggle/input. Attach a Kaggle image dataset first.')

if MANIFEST.exists():
    manifest = pd.read_csv(MANIFEST)
elif (OUTPUT_DIR / 'hf_pairs.csv').exists():
    manifest = pd.read_csv(OUTPUT_DIR / 'hf_pairs.csv')
    print('Reusing downloaded image-caption manifest:', OUTPUT_DIR / 'hf_pairs.csv')
elif AUTO_MANIFEST_IF_MISSING:
    manifest = build_auto_manifest()
else:
    raise FileNotFoundError(f'MANIFEST does not exist: {MANIFEST}')
required = {'image', 'caption'}
missing = required - set(manifest.columns)
if missing:
    raise ValueError(f'Manifest is missing columns: {sorted(missing)}')
manifest = manifest[['image', 'caption']].dropna().drop_duplicates('image').reset_index(drop=True)
if len(manifest) < MIN_IMAGES:
    raise ValueError(f'Found {len(manifest):,} unique images; this notebook requires at least {MIN_IMAGES:,}.')
def resolve_image(value):
    path = Path(str(value))
    return path if path.is_absolute() else DATA_ROOT / path
manifest['resolved_image'] = manifest['image'].map(resolve_image)
missing_paths = manifest.loc[~manifest['resolved_image'].map(Path.exists)]
if len(missing_paths):
    raise FileNotFoundError(f'{len(missing_paths):,} manifest image paths do not exist; first: {missing_paths.iloc[0].image}')
print(f'{len(manifest):,} image-caption pairs ready')
print(manifest.head(3).to_string(index=False))

In [ ]:
class ImageCaptionDataset(Dataset):
    def __init__(self, frame, transform, tokenizer):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.resolved_image) as image:
            image = self.transform(image.convert('RGB'))
        text = self.tokenizer([str(row.caption)])[0]
        return image, text

preprocess_model, preprocess_train, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=None, force_image_size=IMAGE_SIZE, device='cpu')
del preprocess_model
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
dataset = ImageCaptionDataset(manifest, preprocess_train, tokenizer)
print('Dataset size:', len(dataset))

## Distributed contrastive training

Accelerate launches one process per T4. Features are gathered across both GPUs before the symmetric InfoNCE loss, so the effective batch is `BATCH_PER_GPU * GPU_COUNT`. Checkpoints are written after every epoch and can be resumed by setting `RESUME_CHECKPOINT`.

In [ ]:
from accelerate.utils import set_seed
from open_clip.loss import ClipLoss


def train():
    accelerator = Accelerator(
        mixed_precision="fp16",
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    )
    set_seed(SEED)
    model, _, _ = open_clip.create_model_and_transforms(
        MODEL_NAME, pretrained=None, force_image_size=IMAGE_SIZE,
        device="cpu", output_dict=False,
    )
    with torch.no_grad():
        model.logit_scale.fill_(math.log(1 / TEMPERATURE))
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.98),
    )
    workers = min(NUM_WORKERS, 2)
    loader = DataLoader(
        dataset, batch_size=BATCH_PER_GPU, shuffle=True,
        num_workers=workers, pin_memory=True,
        persistent_workers=workers > 0, drop_last=True,
    )
    model, optimizer, loader = accelerator.prepare(model, optimizer, loader)
    if len(loader) == 0:
        raise ValueError("Dataset is too small for the configured batch size.")
    loss_fn = ClipLoss(
        local_loss=True, gather_with_grad=True, cache_labels=True,
        rank=accelerator.process_index, world_size=accelerator.num_processes,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS * math.ceil(len(loader) / GRAD_ACCUM_STEPS),
    )
    accelerator.print(
        f"Training on {accelerator.num_processes} GPU(s); "
        f"{len(loader)} batches/epoch; {EPOCHS} epochs.", flush=True,
    )
    start = time.perf_counter()
    global_step = 0
    optimizer.zero_grad(set_to_none=True)
    for epoch in range(EPOCHS):
        model.train()
        running_loss = torch.zeros((), device=accelerator.device)
        for images, texts in loader:
            with accelerator.accumulate(model):
                with accelerator.autocast():
                    image_features, text_features, logit_scale = model(images, texts)
                # OpenCLIP already exponentiates the returned logit_scale.
                loss = loss_fn(
                    image_features.float(), text_features.float(), logit_scale.float(),
                )
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                if accelerator.sync_gradients and not accelerator.optimizer_step_was_skipped:
                    scheduler.step()
                    with torch.no_grad():
                        accelerator.unwrap_model(model).logit_scale.clamp_(0, math.log(100))
                optimizer.zero_grad(set_to_none=True)
            running_loss += loss.detach()
            global_step += 1
            if global_step == 1 or global_step % 20 == 0:
                elapsed = time.perf_counter() - start
                remaining = elapsed / global_step * (EPOCHS * len(loader) - global_step)
                accelerator.print(
                    f"epoch={epoch + 1}/{EPOCHS} batch={global_step} "
                    f"loss={loss.item():.4f} ETA={remaining / 3600:.2f}h", flush=True,
                )
        mean_loss = accelerator.reduce(running_loss / len(loader), reduction="mean").item()
        accelerator.wait_for_everyone()
        if accelerator.is_main_process:
            checkpoint = {
                "model": accelerator.unwrap_model(model).state_dict(),
                "model_name": MODEL_NAME, "image_size": IMAGE_SIZE, "epoch": epoch + 1,
            }
            # Replace one checkpoint each epoch to limit disk use.
            torch.save(checkpoint, OUTPUT_DIR / "checkpoint_latest.pt")
            if epoch + 1 == EPOCHS:
                torch.save(checkpoint, OUTPUT_DIR / "forensix_clip_final.pt")
            print(f"Epoch {epoch + 1}: mean loss={mean_loss:.4f}; checkpoint saved.", flush=True)
        accelerator.wait_for_everyone()
    accelerator.print("Saved:", OUTPUT_DIR / "forensix_clip_final.pt", flush=True)

# This cell only defines train(). Launch it from Cell 6.


## Cell 6 — Launch training and record validation metrics


In [ ]:
import inspect, subprocess, sys, textwrap

if (OUTPUT_DIR / "forensix_clip_final.pt").exists():
    raise RuntimeError("A final checkpoint already exists. Training was not restarted. Use a new OUTPUT_DIR for a new run.")

# Use the downloaded images already in manifest.
frame = manifest.copy().reset_index(drop=True)
frame["resolved_image"] = frame["resolved_image"].astype(str)
VAL_PAIRS = 1000
if len(frame) <= VAL_PAIRS:
    raise ValueError("Need more than 1,000 pairs for training plus validation.")
if frame["resolved_image"].duplicated().any():
    raise ValueError("Deduplicate image paths before splitting.")
val_frame = frame.sample(n=VAL_PAIRS, random_state=SEED)
train_frame = frame.drop(val_frame.index).reset_index(drop=True)
val_frame = val_frame.reset_index(drop=True)
train_frame.to_csv(OUTPUT_DIR / "train_pairs.csv", index=False)
val_frame.to_csv(OUTPUT_DIR / "validation_pairs.csv", index=False)

training_source = textwrap.dedent(inspect.getsource(train))
anchor = "            checkpoint = {"
if training_source.count(anchor) != 1:
    raise RuntimeError("Use the most recent Cell 5 from this chat first.")
training_source = training_source.replace(
    anchor,
    "            record_epoch(epoch + 1, mean_loss, model, time.perf_counter() - start)\n" + anchor,
)
config = {name: globals()[name] for name in (
    "MODEL_NAME", "IMAGE_SIZE", "EPOCHS", "BATCH_PER_GPU",
    "GRAD_ACCUM_STEPS", "LR", "WEIGHT_DECAY", "TEMPERATURE",
    "NUM_WORKERS", "SEED",
)}
config["OUTPUT_DIR"] = str(OUTPUT_DIR)

helpers = r'''
@torch.inference_mode()
def record_epoch(epoch, train_loss, wrapped_model, elapsed):
    model = wrapped_model.module if hasattr(wrapped_model, "module") else wrapped_model
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device
    image_vectors, text_vectors = [], []
    for images, texts in validation_loader:
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            image_vectors.append(model.encode_image(images.to(device), normalize=True).float().cpu())
            text_vectors.append(model.encode_text(texts.to(device), normalize=True).float().cpu())
    image_vectors = torch.cat(image_vectors)
    text_vectors = torch.cat(text_vectors)
    similarity = image_vectors @ text_vectors.T
    scale = model.logit_scale.detach().float().exp().cpu()
    labels = torch.arange(len(similarity))
    validation_loss = (
        torch.nn.functional.cross_entropy(scale * similarity, labels)
        + torch.nn.functional.cross_entropy(scale * similarity.T, labels)
    ) / 2
    row = {
        "epoch": epoch, "train_loss": train_loss,
        "validation_loss": validation_loss.item(),
        "temperature": float(1 / scale), "logit_scale": float(scale),
        "elapsed_hours": elapsed / 3600, "validation_pairs": len(labels),
    }
    for direction, scores in (("i2t", similarity), ("t2i", similarity.T)):
        for k in (1, 5, 10):
            matches = scores.topk(min(k, len(labels)), dim=1).indices
            row[f"{direction}_R{k}"] = (
                matches == labels[:, None]
            ).any(dim=1).float().mean().item() * 100
    history.append(row)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
    (OUTPUT_DIR / "training_metrics.json").write_text(json.dumps(history, indent=2))
    np.savez_compressed(
        OUTPUT_DIR / "validation_embeddings.npz",
        image=image_vectors.numpy(), text=text_vectors.numpy(),
    )
    print(
        f"VALIDATION epoch={epoch} loss={row['validation_loss']:.4f} "
        f"temperature={row['temperature']:.5f} "
        f"I2T R@1={row['i2t_R1']:.2f}% T2I R@1={row['t2i_R1']:.2f}%",
        flush=True,
    )
    model.train(was_training)
'''

preamble = '''from pathlib import Path
import os, time, math, json, random
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from accelerate import Accelerator
from accelerate.utils import set_seed
from open_clip.loss import ClipLoss
import open_clip
'''
dataset_source = '''
class ImageCaptionDataset(Dataset):
    def __init__(self, frame, transform, tokenizer):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["resolved_image"]) as image:
            image = self.transform(image.convert("RGB"))
        return image, self.tokenizer([str(row["caption"])])[0]
'''
setup = '''
OUTPUT_DIR = Path(OUTPUT_DIR)
preprocess_model, preprocess_train, preprocess_eval = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=None, force_image_size=IMAGE_SIZE, device="cpu",
)
del preprocess_model
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
dataset = ImageCaptionDataset(pd.read_csv(OUTPUT_DIR / "train_pairs.csv"), preprocess_train, tokenizer)
validation_dataset = ImageCaptionDataset(pd.read_csv(OUTPUT_DIR / "validation_pairs.csv"), preprocess_eval, tokenizer)
validation_loader = DataLoader(validation_dataset, batch_size=64, shuffle=False, num_workers=0)
history = []
'''
script = (
    preamble + f"globals().update(json.loads({json.dumps(config)!r}))\n"
    + dataset_source
    + setup + helpers + "\n" + training_source
    + '\nif __name__ == "__main__":\n    train()\n'
)
script_path = OUTPUT_DIR / "train_with_metrics.py"
compile(script, str(script_path), "exec")
script_path.write_text(script, encoding="utf-8")

# Fresh processes avoid inheriting the notebook's CUDA state.
probe = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
    capture_output=True, text=True, check=True,
)
gpu_count = int(probe.stdout.strip().splitlines()[-1])
if gpu_count < 1:
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings first.")
num_processes = min(2, gpu_count)

env = os.environ.copy()
env["OMP_NUM_THREADS"] = "2"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"
print(f"{len(train_frame):,} training pairs; {len(val_frame):,} held-out pairs; {num_processes} GPUs", flush=True)
command = [
    sys.executable, "-m", "torch.distributed.run", "--standalone",
    f"--nproc_per_node={num_processes}", str(script_path),
]
with (OUTPUT_DIR / "training.log").open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        raise
if returncode:
    raise RuntimeError(f"Training exited with code {returncode}. See {OUTPUT_DIR / 'training.log'}.")
print("Training finished. Run Cell 7 to create the proof plots.")


## Cell 7 — Plot loss, temperature, and held-out retrieval accuracy


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

history_path = OUTPUT_DIR / "training_history.csv"
if not history_path.exists():
    raise FileNotFoundError("Cell 6 writes metrics after each completed epoch.")
history = pd.read_csv(history_path)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].plot(history.epoch, history.train_loss, "o-", label="Train: batch contrastive loss")
axes[0, 0].plot(history.epoch, history.validation_loss, "o-", label="Validation: 1,000-pair contrastive loss")
axes[0, 0].set(title="Loss (different candidate pool sizes)", xlabel="Epoch", ylabel="Loss")
axes[0, 0].legend(fontsize=8)
axes[0, 1].plot(history.epoch, history.temperature, "o-", color="purple")
axes[0, 1].set(title="Learned temperature = 1 / logit scale", xlabel="Epoch", ylabel="Temperature")
for direction, ax, title in (
    ("i2t", axes[1, 0], "Image → text retrieval"),
    ("t2i", axes[1, 1], "Text → image retrieval"),
):
    for k in (1, 5, 10):
        ax.plot(history.epoch, history[f"{direction}_R{k}"], "o-", label=f"Recall@{k}")
    chance = 100 / int(history.validation_pairs.iloc[-1])
    ax.axhline(chance, color="gray", linestyle="--", label=f"R@1 chance: {chance:.2f}%")
    ax.set(title=title, xlabel="Epoch", ylabel="Correct matched-pair retrieval (%)", ylim=(0, 100))
    ax.legend(fontsize=8)
for ax in axes.flat:
    ax.grid(alpha=0.25)
fig.suptitle("ForensiX CLIP — measured training and held-out retrieval", fontsize=15)
fig.tight_layout()
plot_path = OUTPUT_DIR / "training_proof.png"
fig.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()
display(history.round(4))
print("Saved:", plot_path)


## Cell 8 — Display held-out retrieval examples


In [ ]:
import numpy as np
from textwrap import fill

vectors = np.load(OUTPUT_DIR / "validation_embeddings.npz")
validation = pd.read_csv(OUTPUT_DIR / "validation_pairs.csv")
queries = np.random.default_rng(SEED).choice(len(validation), size=min(4, len(validation)), replace=False)
fig, axes = plt.subplots(len(queries), 4, figsize=(16, 4 * len(queries)), squeeze=False)
for row, query in enumerate(queries):
    scores = vectors["image"][query] @ vectors["text"].T
    top3 = np.argsort(-scores)[:3]
    for col, image_index in enumerate([query, *top3]):
        with Image.open(validation.iloc[image_index]["resolved_image"]) as image:
            axes[row, col].imshow(image.convert("RGB"))
        if col == 0:
            title = f"Query image (pair {query})"
        else:
            title = (
                f"Top {col}: matched={image_index == query}\n"
                f"cosine={scores[image_index]:.3f}\n"
                + fill(str(validation.iloc[image_index]["caption"])[:160], 36)
            )
        axes[row, col].set_title(title, fontsize=9)
        axes[row, col].axis("off")
fig.suptitle("Held-out image-to-text results — retrieved captions shown with their source images")
fig.tight_layout()
retrieval_path = OUTPUT_DIR / "retrieval_examples.png"
fig.savefig(retrieval_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", retrieval_path)


## Cell 9 — Export and numerically validate ONNX


In [ ]:
import numpy as np
import onnx
import onnxruntime as ort

checkpoint_path = OUTPUT_DIR / "forensix_clip_final.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
export_model, _, preprocess_eval = open_clip.create_model_and_transforms(
    checkpoint["model_name"], pretrained=None,
    force_image_size=checkpoint["image_size"], device="cpu",
)
export_model.load_state_dict(checkpoint["model"])
export_model.eval()

class ImageEncoder(torch.nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.visual = clip_model.visual

    def forward(self, image):
        return torch.nn.functional.normalize(self.visual(image), dim=-1)

encoder = ImageEncoder(export_model).eval()
size = checkpoint["image_size"]
dummy = torch.randn(1, 3, size, size)
onnx_path = OUTPUT_DIR / "forensix_clip_image_encoder.onnx"
torch.onnx.export(
    encoder, dummy, str(onnx_path), input_names=["image"],
    output_names=["embedding"],
    dynamic_axes={"image": {0: "batch"}, "embedding": {0: "batch"}},
    opset_version=17, do_constant_folding=True, dynamo=False,
)
onnx.checker.check_model(str(onnx_path))
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
validation = pd.read_csv(OUTPUT_DIR / "validation_pairs.csv")
with Image.open(validation.iloc[0]["resolved_image"]) as image:
    sample = preprocess_eval(image.convert("RGB")).unsqueeze(0)
with torch.inference_mode():
    expected = encoder(sample).numpy()
actual = session.run(["embedding"], {"image": sample.numpy()})[0]
np.testing.assert_allclose(actual, expected, rtol=1e-3, atol=1e-4)
np.testing.assert_allclose(np.linalg.norm(actual, axis=1), 1, atol=1e-4)
metadata = {
    "model_name": checkpoint["model_name"], "image_size": size,
    "input_layout": "NCHW", "input_dtype": "float32",
    "output_dimensions": int(actual.shape[1]), "output_normalized": True,
    "preprocessing": open_clip.get_model_preprocess_cfg(export_model),
    "preprocess_transform": str(preprocess_eval),
    "onnx_vs_pytorch_max_absolute_error": float(np.max(np.abs(actual - expected))),
}
(OUTPUT_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2))
print("ONNX structure and real-image numerical checks passed:", onnx_path)


## Cell 10 — Package models, plots, metrics, and integrity hashes


In [ ]:
import hashlib, platform, zipfile
from datetime import datetime, timezone
from importlib.metadata import version
from IPython.display import FileLink

artifacts = [
    "training_history.csv", "training_metrics.json", "training_proof.png",
    "retrieval_examples.png", "validation_embeddings.npz",
    "train_pairs.csv", "validation_pairs.csv", "training.log",
    "train_with_metrics.py", "model_metadata.json",
    "forensix_clip_final.pt", "forensix_clip_image_encoder.onnx",
]
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

missing = [name for name in artifacts if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Run the preceding cells first. Missing: {missing}")
report = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "training": "random initialization; pretrained=None",
    "validation": "seeded held-out pairs; evaluation transform; strict matching by row index",
    "retrieval_candidate_pool": len(pd.read_csv(OUTPUT_DIR / "validation_pairs.csv")),
    "python": platform.python_version(),
    "packages": {name: version(name) for name in ("torch", "open_clip_torch", "accelerate")},
    "results": pd.read_csv(OUTPUT_DIR / "training_history.csv").iloc[-1].to_dict(),
    "sha256": {name: sha256_file(OUTPUT_DIR / name) for name in artifacts},
}
(OUTPUT_DIR / "proof_manifest.json").write_text(json.dumps(report, indent=2))
bundle_path = OUTPUT_DIR / "forensix_training_results.zip"
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_STORED) as bundle:
    for name in artifacts + ["proof_manifest.json"]:
        bundle.write(OUTPUT_DIR / name, arcname=name)
print("Results bundle:", bundle_path)
display(FileLink(str(bundle_path)))
